In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from utils import preprocess_data, get_features_and_target
from regression_utils import (
    train_and_compare_regression,
    print_results_table
)

In [7]:
#Подготовка данных
df = preprocess_data('data.xlsx')
X, y = get_features_and_target(df, 'IC50, mM', task_type='regression')
print(f"\nЦелевая переменная: IC50 (логарифмированная через log1p)")
print(f"Размер X: {X.shape}, размер y: {y.shape}")
print(f"\nСтатистика y (после log1p):")
print(f"  min:    {y.min():.3f}")
print(f"  max:    {y.max():.3f}")
print(f"  mean:   {y.mean():.3f}")
print(f"  median: {y.median():.3f}")
print(f"  std:    {y.std():.3f}")

Загружено: 1001 объектов, 213 столбцов
Удалено константных признаков: 33
Пропуски заполнены медианой каждого столбца
Удалено признаков с корреляцией > 0.95: 33
Итого: 1001 объектов, 147 столбцов (144 признаков)

Целевая переменная: IC50 (логарифмированная через log1p)
Размер X: (1001, 144), размер y: (1001,)

Статистика y (после log1p):
  min:    0.004
  max:    8.326
  mean:   3.982
  median: 3.863
  std:    1.861


In [3]:
# Запускаем обучение всех моделей с подбором гиперпараметров
results_df, best_models, splits = train_and_compare_regression(
    X, y,
    target_name='IC50',
    cv=5,
    test_size=0.3,
    random_state=42
)

# Распаковываем разбиение на train/test
X_train, X_test, y_train, y_test = splits

In [4]:
#Вывод итоговой матрицы
print_results_table(results_df, target_name='IC50')

# Сохраняем результаты в csv
from pathlib import Path; Path('results').mkdir(parents=True, exist_ok=True)
results_df.to_csv('results/results_regression_IC50.csv', index=False)

Итоговая таблица сравнений для IC50
           Model     CV R2  Train R2  Test R2  Test MAE  Test RMSE
    RandomForest  0.421681  0.844398 0.413083  1.136219   1.439624
             SVR  0.366445  0.597681 0.388104  1.160750   1.469940
GradientBoosting  0.403402  0.770757 0.371799  1.175352   1.489397
             KNN  0.329386  0.576159 0.310328  1.211226   1.560566
           Lasso  0.218814  0.282780 0.249772  1.352255   1.627638
    DecisionTree  0.225995  0.504371 0.205333  1.338124   1.675150
           Ridge  0.087758  0.542728 0.165672  1.339664   1.716443
LinearRegression -0.026430  0.548850 0.069974  1.380576   1.812211

 Лучшая модель: RandomForest (Test R2 = 0.4131) ***


In [5]:
# График сравнения R^2 на тесте для всех моделей
plt.figure(figsize=(12, 6))
sorted_df = results_df.sort_values('Test R2', ascending=True)
plt.barh(sorted_df['Model'], sorted_df['Test R2'], color='steelblue', edgecolor='black')
plt.xlabel('R^2 на тестовой выборке')
plt.title('Сравнение моделей регрессии по R^2 (IC50)')
plt.axvline(x=0, color='black', linestyle='-', linewidth=0.5)
plt.tight_layout()
plt.savefig('results/comparison_IC50.png', dpi=100, bbox_inches='tight')
plt.close()

# График: предсказания vs реальные значения для лучшей модели
best_model_name = results_df.iloc[0]['Model']
best_model = best_models[best_model_name]

y_pred_test = best_model.predict(X_test)

plt.figure(figsize=(8, 8))
plt.scatter(y_test, y_pred_test, alpha=0.5, color='steelblue')
min_val = min(y_test.min(), y_pred_test.min())
max_val = max(y_test.max(), y_pred_test.max())
plt.plot([min_val, max_val], [min_val, max_val], 'r--', label='Идеальное предсказание')
plt.xlabel('Истинное log(1 + IC50)')
plt.ylabel('Предсказанное log(1 + IC50)')
plt.title(f'Предсказания vs реальные значения\n(лучшая модель: {best_model_name})')
plt.legend()
plt.tight_layout()
plt.savefig('results/predictions_IC50.png', dpi=100, bbox_inches='tight')
plt.close()

In [6]:
# Важность признаков
# Извлекаем модель из Pipeline, чтобы получить feature_importances_
final_model = best_model.named_steps['model']
if hasattr(final_model, 'feature_importances_'):
    print(f"\n15 важных признаков для модели {best_model_name}")
    importances = pd.Series(final_model.feature_importances_, index=X.columns)
    top_features = importances.sort_values(ascending=False).head(15)
    for feat, imp in top_features.items():
        print(f"  {feat:30s}: {imp:.4f}")

    # Визуализация
    plt.figure(figsize=(10, 6))
    top_features.sort_values().plot(kind='barh', color='coral', edgecolor='black')
    plt.xlabel('Важность признака')
    plt.title(f'15 признаков по важности для предсказания IC50\n({best_model_name})')
    plt.tight_layout()
    plt.savefig('results/feature_importance_IC50.png', dpi=100, bbox_inches='tight')
    plt.close()


15 важных признаков для модели RandomForest
  VSA_EState8                   : 0.1298
  VSA_EState6                   : 0.0489
  BCUT2D_MRLOW                  : 0.0331
  VSA_EState4                   : 0.0295
  PEOE_VSA1                     : 0.0255
  PEOE_VSA7                     : 0.0251
  MolLogP                       : 0.0243
  NHOHCount                     : 0.0226
  FpDensityMorgan3              : 0.0220
  qed                           : 0.0194
  FpDensityMorgan1              : 0.0191
  BCUT2D_CHGHI                  : 0.0182
  SlogP_VSA5                    : 0.0180
  EState_VSA5                   : 0.0178
  BalabanJ                      : 0.0173
